# 
DSA 210 — Spotify Project
## Notebook 4: Last.fm Mood Tag Enrichment
**Selin Nardal**

### Motivation
Spotify's Web API deprecated the public `audio-features` endpoint in November 2024, removing programmatic access to `valence`, `energy`, and `danceability`.  These features were central to the original H1 and H2 hypotheses.

As an alternative enrichment source, this notebook uses the **Last.fm API**, which exposes community-assigned **tags** for each track (e.g. `chill`, `energetic`, `sad`, `happy`, `melancholic`).  These tags act as a crowd-sourced mood proxy and let us reformulate the hypotheses around tag-based mood categories instead of Spotify's numeric features.

### What this notebook does
1. Loads the cleaned streaming history from notebook 01.
2. Builds a list of unique (artist, track) pairs.
3. Queries the Last.fm `track.getTopTags` endpoint for a sample of tracks.
4. Saves the raw tag responses to disk so we don't re-hit the API on every run.

**Tonight's scope:** small sample (~50 tracks) to confirm the API works and inspect the tag quality.  Full enrichment of all unique tracks will follow in a later commit.

## 1. Setup

In [2]:
import os
import json
import time
from pathlib import Path

import pandas as pd
import requests

DATA_DIR = Path('data')
TAGS_DIR = DATA_DIR / 'lastfm_tags'
TAGS_DIR.mkdir(parents=True, exist_ok=True)

print('Setup complete.')

Setup complete.


## 2. API key
Paste your Last.fm API key below.  This is read-only public data so the key is low-risk, but for cleanliness we'll still keep it out of commits via `.gitignore` later.

In [3]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

LASTFM_API_KEY = os.getenv('LASTFM_API_KEY')

assert LASTFM_API_KEY is not None, 'LASTFM_API_KEY not found. Check your .env file.'
BASE_URL = 'https://ws.audioscrobbler.com/2.0/'

print(f'API key loaded successfully (length: {len(LASTFM_API_KEY)} chars)')


API key loaded successfully (length: 32 chars)


## 3. Load streaming history and build unique track list

In [4]:
df = pd.read_csv(DATA_DIR / 'streaming_clean.csv')
print(f'Total rows: {len(df):,}')

unique_tracks = (
    df[['artist', 'track_name']]
    .dropna()
    .drop_duplicates()
    .rename(columns={'artist': 'artist', 'track_name': 'track'})
    .reset_index(drop=True)
)
print(f'Unique (artist, track) pairs: {len(unique_tracks):,}')
unique_tracks.head()

Total rows: 15,581
Unique (artist, track) pairs: 5,453


,artist,track
0,Tory Lanez,The Color Violet
1,Brent Faiyaz,Poison
2,The Weeknd,Starboy
3,Güneş,Suçlarımdan Biri
4,Yüzyüzeyken Konuşuruz,Boş Gemiler


## 4. Last.fm tag-fetching function

We hit `track.getTopTags`, which returns the most-voted tags for a given (artist, track) pair.  We rate-limit ourselves to ~5 requests/sec to stay polite — Last.fm's published limit is 5/sec averaged.

In [5]:
def fetch_top_tags(artist: str, track: str, api_key: str = LASTFM_API_KEY):
    """Return list of {'name': str, 'count': int} for a track, or [] on miss."""
    params = {
        'method': 'track.getTopTags',
        'artist': artist,
        'track': track,
        'api_key': api_key,
        'format': 'json',
        'autocorrect': 1,  # let Last.fm fix small typos
    }
    try:
        r = requests.get(BASE_URL, params=params, timeout=10)
        r.raise_for_status()
        data = r.json()
    except Exception as e:
        print(f'  ! error for {artist} — {track}: {e}')
        return []
    tags = data.get('toptags', {}).get('tag', [])
    return [{'name': t['name'].lower(), 'count': int(t.get('count', 0))} for t in tags]

# Quick smoke test
demo = fetch_top_tags('The Weeknd', 'Blinding Lights')
print('Demo result (top 10 tags):')
for t in demo[:10]:
    print(f"  {t['name']:<25} {t['count']}")

Demo result (top 10 tags):
  synthwave                 100
  synthpop                  94
  pop                       51
  2019                      37
  the weeknd                12
  2010s                     12
  electropop                12
  -1001740215468            3
  synth-pop                 2
  2020s                     1


## 5. Sample run — first 50 unique tracks

Tonight we only fetch a small sample to verify the pipeline.  Results are cached to `data/lastfm_tags/sample_tags.json` so we don't re-query later.  A full run over all unique tracks will be a separate commit.

In [6]:
SAMPLE_SIZE = 50
OUT_FILE = TAGS_DIR / 'sample_tags.json'

sample = unique_tracks.head(SAMPLE_SIZE)
results = {}

for i, row in sample.iterrows():
    key = f"{row['artist']} ||| {row['track']}"
    print(f"[{i+1}/{SAMPLE_SIZE}] {row['artist']} — {row['track']}")
    results[key] = fetch_top_tags(row['artist'], row['track'])
    time.sleep(0.25)  # ~4 req/sec, stays under Last.fm's 5/sec limit

with open(OUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f'\nSaved {len(results)} tag-sets to {OUT_FILE}')

[1/50] Tory Lanez — The Color Violet
[2/50] Brent Faiyaz — Poison
[3/50] The Weeknd — Starboy
[4/50] Güneş — Suçlarımdan Biri
[5/50] Yüzyüzeyken Konuşuruz — Boş Gemiler
[6/50] Jung Kook — Seven (feat. Latto)
[7/50] Doja Cat — Paint The Town Red
[8/50] Tate McRae — greedy
[9/50] Jung Kook — 3D (feat. Jack Harlow)
[10/50] Drake — IDGAF (feat. Yeat)
[11/50] Taylor Swift — Cruel Summer
[12/50] KAROL G — QLONA
[13/50] Mitski — My Love Mine All Mine
[14/50] Kenya Grace — Strangers
[15/50] Myke Towers — LALA
[16/50] Maluma — Según Quién
[17/50] David Kushner — Daylight
[18/50] Calle 24 — Que Onda
[19/50] cassö — Prada
[20/50] Harry Styles — As It Was
[21/50] SZA — Kill Bill
[22/50] Teoman — Serseri
[23/50] Ufuk Beydemir — Ay Tenli Kadın
[24/50] Şebnem Ferah — Hoşçakal
[25/50] Sufle — Ne Söz Ne Saz
[26/50] Walk off the Earth — My Stupid Heart
[27/50] The Lumineers — just like heaven
[28/50] Walk off the Earth — The Caviar Lime Song
[29/50] Walk off the Earth — Rude
[30/50] Walk off the Earth —

## 6. Quick sanity check — what tags came back?

In [7]:
from collections import Counter

tag_counter = Counter()
hits = 0
for tags in results.values():
    if tags:
        hits += 1
        for t in tags[:5]:  # take top 5 per track
            tag_counter[t['name']] += 1

print(f'Tracks with at least one tag: {hits}/{len(results)}')
print('\nTop 20 tags across the sample:')
for tag, count in tag_counter.most_common(20):
    print(f'  {tag:<25} {count}')

Tracks with at least one tag: 21/50

Top 20 tags across the sample:
  pop                       14
  rnb                       8
  synthpop                  4
  electropop                2
  dance                     2
  my top songs              2
  singer-songwriter         2
  trance                    2
  indie pop                 2
  dream pop                 2
  color                     1
  colors                    1
  colours                   1
  colour                    1
  synthwave                 1
  smooth soul               1
  alternative rnb           1
  dark rnb                  1
  brent faiyaz              1
  2016                      1


## 7. Next steps
- Expand the run to all unique tracks (will take a while — ~7,000 requests at 4/sec ≈ 30 minutes).
- Build a `mood_category` mapping from raw tags to a small controlled vocabulary (e.g. `happy / sad / energetic / chill / aggressive / romantic`).
- Re-formulate H1 and H2 in terms of mood categories and re-test.
- Use mood categories as features in K-Means clustering and Random Forest.

In [8]:

OUT_FILE_FULL = TAGS_DIR / 'all_tags.json'

if OUT_FILE_FULL.exists():
    with open(OUT_FILE_FULL, 'r', encoding='utf-8') as f:
        results_full = json.load(f)
    print(f'Resuming: {len(results_full)} tracks already cached.')
else:
    results_full = {}

total = len(unique_tracks)
to_fetch = []
for _, row in unique_tracks.iterrows():
    key = f"{row['artist']} ||| {row['track']}"
    if key not in results_full:
        to_fetch.append((key, row['artist'], row['track']))

print(f'Will fetch {len(to_fetch):,} new tracks (out of {total:,} total).')
print(f'Estimated time: ~{len(to_fetch) / 4 / 60:.1f} minutes')
print('-' * 50)

SAVE_EVERY = 100

for i, (key, artist, track) in enumerate(to_fetch, start=1):
    results_full[key] = fetch_top_tags(artist, track)
    
    if i % 50 == 0:
        hits_so_far = sum(1 for v in results_full.values() if v)
        print(f'  [{i}/{len(to_fetch)}] hits={hits_so_far}/{len(results_full)} ({hits_so_far/len(results_full)*100:.1f}%)')
    
    if i % SAVE_EVERY == 0:
        with open(OUT_FILE_FULL, 'w', encoding='utf-8') as f:
            json.dump(results_full, f, ensure_ascii=False)
    
    time.sleep(0.25)

with open(OUT_FILE_FULL, 'w', encoding='utf-8') as f:
    json.dump(results_full, f, ensure_ascii=False)

hits_total = sum(1 for v in results_full.values() if v)
print(f'\nDone! {hits_total:,}/{len(results_full):,} tracks have tags ({hits_total/len(results_full)*100:.1f}%).')
print(f'Saved to: {OUT_FILE_FULL}')


Will fetch 5,453 new tracks (out of 5,453 total).
Estimated time: ~22.7 minutes
--------------------------------------------------
  [50/5453] hits=21/50 (42.0%)
  [100/5453] hits=27/100 (27.0%)
  [150/5453] hits=36/150 (24.0%)
  [200/5453] hits=41/200 (20.5%)
  [250/5453] hits=45/250 (18.0%)
  [300/5453] hits=56/300 (18.7%)
  [350/5453] hits=65/350 (18.6%)
  [400/5453] hits=73/400 (18.2%)
  [450/5453] hits=87/450 (19.3%)
  [500/5453] hits=88/500 (17.6%)
  [550/5453] hits=101/550 (18.4%)
  [600/5453] hits=127/600 (21.2%)
  [650/5453] hits=127/650 (19.5%)
  [700/5453] hits=150/700 (21.4%)
  [750/5453] hits=169/750 (22.5%)
  [800/5453] hits=180/800 (22.5%)
  [850/5453] hits=208/850 (24.5%)
  [900/5453] hits=213/900 (23.7%)
  [950/5453] hits=228/950 (24.0%)
  [1000/5453] hits=260/1000 (26.0%)
  [1050/5453] hits=283/1050 (27.0%)
  [1100/5453] hits=295/1100 (26.8%)
  [1150/5453] hits=298/1150 (25.9%)
  [1200/5453] hits=301/1200 (25.1%)
  [1250/5453] hits=330/1250 (26.4%)
  [1300/5453] hits=